# 끊긴 가격을 이어 붙인다 — 액면분할·병합·권리락

장환님이 피처를 검증하다 **"삼성전자 5일 수익률이 −98%"** 를 발견해 전달해 주신 건을
전수로 재고 고친 과정입니다.

이 노트북이 답하는 것

1. 정말로 −98% 가 나오는가 — **재현**
2. 얼마나 많은가 — **전수**
3. 🔴 왜 방식이 하나로는 안 되는가 — **거래정지 재개일**
4. 고치면 무엇이 달라지는가 — **효과**

> **읽는 법**: 위에서부터 그냥 실행하면 됩니다. DB 는 **읽기 전용**으로만 엽니다.

In [1]:
import sqlite3
import sys
from fractions import Fraction
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():      # 노트북을 어디서 열든 루트를 찾는다
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from common import corporate_actions as ca  # noqa: E402

DB = f"file:{(ROOT / 'data' / 'krx_cache.db').as_posix()}?mode=ro"   # 읽기 전용
con = sqlite3.connect(DB, uri=True)
con.execute("PRAGMA cache_size = -300000")

COLS = ["bas_dd", "open", "high", "low", "close", "change",
        "change_rate", "volume", "listed_shares"]

def series(code):
    q = f"SELECT {','.join(COLS)} FROM daily_price WHERE code=? ORDER BY bas_dd"
    return [dict(zip(COLS, r, strict=True)) for r in con.execute(q, (code,))]

print("daily_price", f"{con.execute('SELECT COUNT(*) FROM daily_price').fetchone()[0]:,}행")

daily_price 9,220,879행


---
## 1. 재현 — 삼성전자 2018-05-04

액면분할 50:1 이 있던 날입니다. **분할 앞 3거래일이 거래정지**라는 점을 눈여겨봐 주세요.
나중에 이것이 핵심이 됩니다.

In [2]:
sam = pd.DataFrame(series("005930"))
win = sam[sam.bas_dd.between("20180425", "20180510")].copy()
win["정지"] = win.apply(lambda r: "■" if r.open == 0 or r.volume == 0 else "", axis=1)
win[["bas_dd", "close", "change", "change_rate", "volume", "listed_shares", "정지"]]

,bas_dd,close,change,change_rate,volume,listed_shares,정지
2054,20180425,2520000,-3000,-0.12,332292,128386494,
2055,20180426,2607000,87000,3.45,360931,128386494,
2056,20180427,2650000,43000,1.65,606216,128386494,
2057,20180430,2650000,0,0.00,0,128386494,■
2058,20180502,2650000,0,0.00,0,128386494,■
2059,20180503,2650000,0,0.00,0,128386494,■
2060,20180504,51900,-1100,-2.08,39565391,6419324700,
2061,20180508,52600,700,1.35,23104720,6419324700,
2062,20180509,50900,-1700,-3.23,16128305,6419324700,
2063,20180510,51600,700,1.38,13905263,6419324700,


In [3]:
전, 후 = 2_650_000, 51_900
print(f"close 로 계산한 수익률 : {(후 / 전 - 1) * 100:>8.2f} %")
print(f"KRX 가 준 등락률       : {-2.08:>8.2f} %")
print()
print("→ 같은 날을 두고 96%p 가 벌어진다. 앞의 값이 파생 피처와 라벨로 전파된다.")

close 로 계산한 수익률 :   -98.04 %
KRX 가 준 등락률       :    -2.08 %

→ 같은 날을 두고 96%p 가 벌어진다. 앞의 값이 파생 피처와 라벨로 전파된다.


### 🟢 단서 — KRX 는 이미 답을 준다

`change`(전일대비)를 보면 분할일에 **−1,100** 입니다. 종가 51,900 에서 이걸 빼면
**53,000** 이고, 이는 2,650,000 ÷ 50 과 정확히 같습니다.

즉 KRX 는 분할 배율을 알고 있고 **`change` 에 담아 줍니다.**

In [4]:
기준가 = 51_900 - (-1_100)
print(f"기준가 = close - change = {기준가:,}")
print(f"기준가 / 전일종가       = {Fraction(기준가, 2_650_000)}   ← 정확히 1/50")
print()
print("close·change 는 둘 다 INTEGER 라 이 계수에는 부동소수 오차가 없다.")

기준가 = close - change = 53,000
기준가 / 전일종가       = 1/50   ← 정확히 1/50

close·change 는 둘 다 INTEGER 라 이 계수에는 부동소수 오차가 없다.


---
## 2. 얼마나 많은가 — 전수

가격이 끊긴 날을 **기준가가 전일종가와 다른 행**으로 셉니다.

In [5]:
adj = pd.read_sql_query('''
    WITH t AS (
      SELECT code, name, bas_dd, open, high, low, close, change, volume, listed_shares,
             LAG(close)         OVER w AS prev_close,
             LAG(listed_shares) OVER w AS prev_shares,
             LAG(open)          OVER w AS prev_open,
             LAG(high)          OVER w AS prev_high,
             LAG(low)           OVER w AS prev_low
      FROM daily_price
      WINDOW w AS (PARTITION BY code ORDER BY bas_dd)
    )
    SELECT * FROM t
    WHERE prev_close > 0 AND close IS NOT NULL AND change IS NOT NULL
      AND (close - change) != prev_close
''', con)

adj["계수"] = (adj.close - adj.change) / adj.prev_close
adj["재개일"] = (adj.prev_open == 0) & (adj.prev_high == 0) & (adj.prev_low == 0)
adj["주식수배율"] = adj.listed_shares / adj.prev_shares

print(f"기준가가 끊긴 행 : {len(adj):,}행 · {adj.code.nunique():,}종")
print(f"계수 범위        : 최소 {adj.계수.min():.4f}배 · "
      f"중앙 {adj.계수.median():.4f}배 · 최대 {adj.계수.max():.1f}배")

기준가가 끊긴 행 : 5,246행 · 2,181종
계수 범위        : 최소 0.0019배 · 중앙 0.9542배 · 최대 120.0배


In [6]:
라벨 = ["×0.5 미만 (분할·무상증자)", "×0.5~0.9", "×0.9~0.99 (권리락)",
        "×0.99~1.01 (미세)", "×1.01~1.5", "×1.5 초과 (병합·감자)"]
구간 = pd.cut(adj.계수, [0, .5, .9, .99, 1.01, 1.5, 1e9], labels=라벨)
pd.DataFrame({"건수": 구간.value_counts().sort_index()}).assign(
    비율=lambda d: (d.건수 / len(adj) * 100).round(1).astype(str) + "%")

,건수,비율
계수,,
×0.5 미만 (분할·무상증자),744,14.2%
×0.5~0.9,1144,21.8%
×0.9~0.99 (권리락),1552,29.6%
×0.99~1.01 (미세),502,9.6%
×1.01~1.5,330,6.3%
×1.5 초과 (병합·감자),974,18.6%


In [7]:
월별 = adj.bas_dd.str[4:6].value_counts().sort_index()
pd.DataFrame({"건수": 월별, "그래프":월별.map(lambda n: "█" * int(n / 월별.max() * 30))})

,건수,그래프
bas_dd,,
01,323,█████████
02,252,███████
03,351,██████████
04,407,███████████
05,723,████████████████████
06,393,███████████
07,400,███████████
08,340,█████████
09,377,██████████


**12월에 몰립니다.** 12월 결산법인의 주식배당 권리락입니다.
이건 `listed_shares` 로는 잡을 수 없습니다 — 주식배당은 주식수가 안 변하니까요.

---
## 3. 🔴 왜 한 방식으로는 안 되는가

기준가만 믿으면 되는 것처럼 보입니다. 그런데 **끊긴 행의 39%가 자본변동이 아닙니다.**

In [8]:
n_재개 = int(adj.재개일.sum())
print(f"기준가가 끊긴 {len(adj):,}행 중")
print(f"  직전이 거래정지(재개일) : {n_재개:,}행 ({n_재개 / len(adj) * 100:.1f}%)")
print()
print("재개일 중 주식수가 거의 안 변한 사례 — 자본변동이 아닌데 계수가 크다:")
가짜 = adj[adj.재개일 & adj.주식수배율.between(0.95, 1.05) & (adj.계수.abs() > 5)]
가짜[["code", "name", "bas_dd", "prev_close", "close", "계수", "주식수배율"]].nlargest(6, "계수")

기준가가 끊긴 5,246행 중
  직전이 거래정지(재개일) : 2,057행 (39.2%)

재개일 중 주식수가 거의 안 변한 사례 — 자본변동이 아닌데 계수가 크다:


,code,name,bas_dd,prev_close,close,계수,주식수배율
3321,111610,승화프리텍,20150817,145,17400,120.00000,0.981781
2512,071970,STX중공업,20170330,2765,55300,20.00000,1.032123
3880,192410,감마누,20200818,408,6240,14.95098,0.974963


정지 중 종가는 직전 값을 **붙들고 있고**, 재개일에 KRX 는 단일가로 기준가를 **새로 잡습니다.**
그 기준가를 조정계수로 쓰면 자본변동이 아닌 것을 조정으로 오해합니다.

위 첫 줄을 그대로 믿으면 **과거 전체가 120배 부풀어납니다.**

### ⚠️ 그렇다고 재개일을 버리면 분할을 전부 잃습니다

In [9]:
진짜 = adj[~adj.재개일]
분할스러운것 = 진짜[(진짜.주식수배율 >= 1.5) | (진짜.주식수배율 <= 1/1.5)]
print(f"거래정지일을 뺀 '진짜 자본변동' : {len(진짜):,}건")
print(f"  그중 액면분할·병합으로 볼 것  : {len(분할스러운것):,}건")
print()
print("→ 분할·병합은 전부 재개일에 있다. KRX 가 주권 교체 때문에 반드시 정지시키기 때문이다.")
print("   삼성전자도 20180430~0503 이 정지였다 (1절 표의 ■ 표시).")

거래정지일을 뺀 '진짜 자본변동' : 3,189건
  그중 액면분할·병합으로 볼 것  : 1건

→ 분할·병합은 전부 재개일에 있다. KRX 가 주권 교체 때문에 반드시 정지시키기 때문이다.
   삼성전자도 20180430~0503 이 정지였다 (1절 표의 ■ 표시).


### 그래서 자리별로 나눠 씁니다

| 유형 | 재는 법 |
|---|---|
| 평상일 | `기준가 = close − change` |
| **재개일** | **상장주식수 배율** (≥1.5배 움직였을 때만) |

---
## 4. 함수를 써 봅니다

`common/corporate_actions.py` 에 넣은 것들입니다.

In [10]:
rows = series("005930")
factors = ca.factor_series(rows)          # 행마다 조정 배율 (Fraction)
adjclose = ca.back_adjusted_closes(rows)  # 후방조정 종가
idx = {r["bas_dd"]: i for i, r in enumerate(rows)}

pd.DataFrame([{
    "날짜": d, "원종가": rows[idx[d]]["close"],
    "계수": str(factors[idx[d]]), "수정주가": round(adjclose[idx[d]], 2),
} for d in ["20180427", "20180430", "20180503", "20180504", "20180508"]])

,날짜,원종가,계수,수정주가
0,20180427,2650000,1,53000.0
1,20180430,2650000,1,53000.0
2,20180503,2650000,1,53000.0
3,20180504,51900,1/50,51900.0
4,20180508,52600,1,52600.0


In [11]:
첫, 끝 = rows[0], rows[-1]
print(f"첫날   {첫['bas_dd']} : 원종가 {첫['close']:>9,}"
      f" → 수정주가 {adjclose[0]:>10,.2f}")
print(f"마지막 {끝['bas_dd']} : 원종가 {끝['close']:>9,}"
      f" → 수정주가 {adjclose[-1]:>10,.2f}")
print()
print("후방조정이라 마지막이 기준점이다 — 원종가와 같아야 정상.")

첫날   20100104 : 원종가   809,000 → 수정주가  16,180.00
마지막 20260831 : 원종가   260,000 → 수정주가 260,000.00

후방조정이라 마지막이 기준점이다 — 원종가와 같아야 정상.


### 🔴 학습에는 `span_factor` 를 씁니다 — 재현성 때문에

후방조정은 **새 분할이 하나 생기면 과거 전체 값이 바뀝니다.** 어제 돌린 학습 결과와
오늘 돌린 결과가 달라지고 **에러는 나지 않습니다.**

`span_factor` 는 **구간 안의 조정만** 보므로 그 문제가 없습니다.

In [12]:
e, x = idx["20180427"], idx["20180508"]
span = ca.span_factor(factors, e, x)

raw = rows[x]["open"] / rows[e]["open"] - 1
fix = rows[x]["open"] / (rows[e]["open"] * span) - 1

print(f"진입 20180427 시가 {rows[e]['open']:>10,}")
print(f"청산 20180508 시가 {rows[x]['open']:>10,}")
print(f"span_factor        {span:>10.6f}   (1/50 = 0.02)")
print()
print(f"  조정 안 함 : {raw * 100:>8.2f} %   ← 틀림")
print(f"  조정 함    : {fix * 100:>8.2f} %   ← 옳음")

진입 20180427 시가  2,669,000
청산 20180508 시가     52,600
span_factor          0.020000   (1/50 = 0.02)

  조정 안 함 :   -98.03 %   ← 틀림
  조정 함    :    -1.46 %   ← 옳음


---
## 5. 효과 — 전 종목

`ret_1` 이 −50% 아래인 행이 얼마나 줄어드는지 셉니다.

In [13]:
codes = [c for (c,) in con.execute("SELECT DISTINCT code FROM daily_price ORDER BY code")]
raw_bad, adj_bad, worst_raw, worst_adj = 0, 0, [], []

for code in codes:
    rs = series(code)
    if len(rs) < 2:
        continue
    a = ca.back_adjusted_closes(rs)
    for i in range(1, len(rs)):
        if rs[i - 1]["close"]:
            r = rs[i]["close"] / rs[i - 1]["close"] - 1
            raw_bad += r <= -0.5
            worst_raw.append((r, code, rs[i]["bas_dd"]))
        if a[i - 1]:
            v = a[i] / a[i - 1] - 1
            adj_bad += v <= -0.5
            worst_adj.append((v, code, rs[i]["bas_dd"]))

print("ret_1 ≤ −50% 인 행")
print(f"  미조정 close : {raw_bad:,} 행")
print(f"  조정   close : {adj_bad:,} 행")
print(f"  제거된 것    : {raw_bad - adj_bad:,} 행 ({(raw_bad - adj_bad) / raw_bad * 100:.1f}%)")

ret_1 ≤ −50% 인 행
  미조정 close : 1,355 행
  조정   close : 745 행
  제거된 것    : 610 행 (45.0%)


In [14]:
worst_raw.sort()
worst_adj.sort()
pd.DataFrame({
    "미조정 최악": [f"{c} {d}  {r:.4f}" + ("  ← 삼성전자" if c in ("005930", "005935") else "")
                    for r, c, d in worst_raw[:8]],
    "조정후 최악": [f"{c} {d}  {r:.4f}" + ("  ← 삼성전자" if c in ("005930", "005935") else "")
                    for r, c, d in worst_adj[:8]],
})

,미조정 최악,조정후 최악
0,008080 20130822 -0.9981,008080 20130822 -1.0000
1,150840 20260115 -0.9841,150840 20260115 -0.9841
2,152550 20221214 -0.9833,057880 20260113 -0.9829
3,057880 20260113 -0.9829,065560 20240718 -0.9796
4,055250 20100917 -0.9808,222810 20260305 -0.9796
5,005935 20180504 -0.9807 ← 삼성전자,263540 20241028 -0.9790
6,005930 20180504 -0.9804 ← 삼성전자,268600 20250225 -0.9786
7,065560 20240718 -0.9796,136510 20240716 -0.9784


**삼성전자·삼성전자우 20180504 가 최악 목록에서 사라졌습니다** — 장환님이 발견한 그 자리입니다.

남은 745행은 **장기 거래정지 뒤 재개**입니다. 828~1,131 거래일 정지 뒤 재개가 상위를 차지하는데,
이건 수정주가가 아니라 **표본 선택**으로 다뤄야 하는 것들입니다.

---
## 6. 정리

| 한 일 | 결과 |
|---|---|
| 가격이 끊긴 날을 전수로 셈 | 5,246행 · 2,181종 (전체의 0.057%) |
| 그중 거래정지 재개일 | 2,057행 (39.2%) — **자본변동이 아니다** |
| 평상일은 기준가 · 재개일은 주식수로 나눠 잼 | `ret_1 ≤ −50%` 1,355 → **745** (45% 감소) |
| 삼성전자 −98% | **사라짐** |

### ⚠️ 아직 적용하지 않은 곳

수익률을 계산하는 자리가 4곳이고 전부 아직 조정을 반영하지 않습니다.
파트 경계를 넘으므로 팀에 묻고 나서 적용합니다.

- `evaluation/horizon.py` 3곳 (백테스트 파트)
- `scripts/export_team_dataset.py::forward_returns_aligned` (데이터 파트)

### 못 잰 것

- 2026년 기준가 조정이 다른 해보다 많은 **이유**
- 남은 745행을 표본에서 뺄지
- `SHARE_RATIO_MIN = 1.5` 문턱의 사이 구간(1.1~2.0)에 무엇이 있는지
- 배당 조정(total return) — 지금은 price return 만

관련: [#51](https://github.com/devlee328288/Alpha_Stack/issues/51) ·
[#44](https://github.com/devlee328288/Alpha_Stack/issues/44) ·
[#38](https://github.com/devlee328288/Alpha_Stack/issues/38)

In [15]:
con.close()
print("DB 닫음 — 이 노트북은 아무것도 쓰지 않았습니다.")

DB 닫음 — 이 노트북은 아무것도 쓰지 않았습니다.
